# 第5章 電波マップ入門

書籍『Sionna RTによる電波伝搬レイトレーシング入門』第5章のコードである。
セルは本文の掲載順に並べてあり、コードセルの中身は本文に印刷したものと同じである。

先頭の「準備」セルだけは本文に印刷していない。本文のコードが前提にしている
import とシーンの読み込みをここで済ませてある。準備セルを実行すれば、以降は
上から順に実行できる。

ただし、本文が説明のために示している断片（自分で作ったシーンのパスや、その
シーンには無い物体名を使う例）はそのままでは動かない。該当するセルの前に注記を
置いてある。

## 準備

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

## 2. 本章で使う都市シーン

In [ ]:
from sionna.rt import load_scene, scene as scene_assets

scene = load_scene(scene_assets.munich)
scene.frequency = 3.5e9

## 3. 受信電力を面で見る

In [ ]:
from sionna.rt import PlanarArray, Transmitter, RadioMapSolver

scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-30.0, 38.0, 35.0]))

solver = RadioMapSolver()
rm = solver(scene, max_depth=3,
            cell_size=[3.0, 3.0],
            samples_per_tx=10**6)
print('Radio map shape:', rm.path_gain.shape)

## 7. 送信機 1 台の電波マップ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rm = solver(scene, max_depth=3, cell_size=[3.0, 3.0],
            samples_per_tx=10**6)
rss_dbm = 10*np.log10(rm.rss.numpy()) + 30
plt.imshow(rss_dbm[0], origin='lower', cmap='jet',
           vmin=-75, vmax=-25)
plt.colorbar(label='RSS [dBm]')

## 8. 複数送信機の電波マップ

In [ ]:
scene.remove('tx0')  # 1 台構成で置いた送信機を外す

tx_positions = [[-120.0, 60.0, 35.0],
                [  90.0, 60.0, 35.0],
                [ -15.0,-125.0, 35.0]]
for i, pos in enumerate(tx_positions):
    scene.add(Transmitter(f'tx{i}', position=pos))

rm = solver(scene, max_depth=3, cell_size=[3.0, 3.0],
            samples_per_tx=10**6)
print('Number of Tx in rm:', rm.path_gain.shape[0])

## 9. 基地局アソシエーション

In [ ]:
import numpy as np
best_tx = np.argmax(rm.rss.numpy(), axis=0)  # 各セルで最強の Tx インデックス

## 10. SINR マップを作る

In [ ]:
rss = rm.rss.numpy()                      # (num_tx, H, W)
signal = rss.max(axis=0)
interf  = rss.sum(axis=0) - signal
N0 = 1e-13                                # 例: -100 dBm
sinr_db = 10*np.log10(signal/(interf + N0))

## 11. カバレッジ率を評価する

In [ ]:
threshold = 0.0  # SINR >= 0 dB をカバレッジとみなす
coverage = (sinr_db >= threshold).mean()
print(f'Coverage at {threshold:.0f} dB: {coverage*100:.1f} %')

## 12. CDF で結果を見る

In [ ]:
signal_dbm = 10*np.log10(np.maximum(signal, 1e-30)) + 30
valid = np.isfinite(sinr_db) & (signal_dbm >= -110.0)
sorted_sinr = np.sort(sinr_db[valid].flatten())
cdf = np.arange(1, len(sorted_sinr) + 1)/len(sorted_sinr)
plt.plot(sorted_sinr, cdf)

## 13. 電波マップからユーザ位置をサンプリングする

In [ ]:
path_gain = rm.path_gain.numpy()[0].flatten()
valid_idx = np.flatnonzero(np.isfinite(path_gain) & (path_gain > 0))
idx = np.random.choice(valid_idx, size=50, replace=True)

centers = rm.cell_centers.numpy().reshape(-1, 3)
positions = centers[idx]  # 選んだセル中心の実空間座標 [m]